In [1]:
# ════════════════════════════════════════════════════════════════════════════
# CELL 1 — LOAD ROUND 5 DATA
# ════════════════════════════════════════════════════════════════════════════

from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROUND = 5
DAYS = [2, 3, 4]

DATA_DIR = Path("Data/round5")

ALGO_PRODUCTS = [
    # Galaxy Sounds Recorders
    "GALAXY_SOUNDS_DARK_MATTER",
    "GALAXY_SOUNDS_BLACK_HOLES",
    "GALAXY_SOUNDS_PLANETARY_RINGS",
    "GALAXY_SOUNDS_SOLAR_WINDS",
    "GALAXY_SOUNDS_SOLAR_FLAMES",

    # Vertical Sleeping Pods
    "SLEEP_POD_SUEDE",
    "SLEEP_POD_LAMB_WOOL",
    "SLEEP_POD_POLYESTER",
    "SLEEP_POD_NYLON",
    "SLEEP_POD_COTTON",

    # Organic Microchips
    "MICROCHIP_CIRCLE",
    "MICROCHIP_OVAL",
    "MICROCHIP_SQUARE",
    "MICROCHIP_RECTANGLE",
    "MICROCHIP_TRIANGLE",

    # Purification Pebbles
    "PEBBLES_XS",
    "PEBBLES_S",
    "PEBBLES_M",
    "PEBBLES_L",
    "PEBBLES_XL",

    # Domestic Robots
    "ROBOT_VACUUMING",
    "ROBOT_MOPPING",
    "ROBOT_DISHES",
    "ROBOT_LAUNDRY",
    "ROBOT_IRONING",

    # UV-Visors
    "UV_VISOR_YELLOW",
    "UV_VISOR_AMBER",
    "UV_VISOR_ORANGE",
    "UV_VISOR_RED",
    "UV_VISOR_MAGENTA",

    # Instant Translators
    "TRANSLATOR_SPACE_GRAY",
    "TRANSLATOR_ASTRO_BLACK",
    "TRANSLATOR_ECLIPSE_CHARCOAL",
    "TRANSLATOR_GRAPHITE_MIST",
    "TRANSLATOR_VOID_BLUE",

    # Construction Panels
    "PANEL_1X2",
    "PANEL_2X2",
    "PANEL_1X4",
    "PANEL_2X4",
    "PANEL_4X4",

    # Liquid Breath Oxygen Shakes
    "OXYGEN_SHAKE_MORNING_BREATH",
    "OXYGEN_SHAKE_EVENING_BREATH",
    "OXYGEN_SHAKE_MINT",
    "OXYGEN_SHAKE_CHOCOLATE",
    "OXYGEN_SHAKE_GARLIC",

    # Protein Snack Packs
    "SNACKPACK_CHOCOLATE",
    "SNACKPACK_VANILLA",
    "SNACKPACK_PISTACHIO",
    "SNACKPACK_STRAWBERRY",
    "SNACKPACK_RASPBERRY",
]

POSITION_LIMITS = {product: 10 for product in ALGO_PRODUCTS}

prices_parts = []
trades_parts = []

for day in DAYS:
    price_path = DATA_DIR / f"prices_round_{ROUND}_day_{day}.csv"
    trade_path = DATA_DIR / f"trades_round_{ROUND}_day_{day}.csv"

    p = pd.read_csv(price_path, sep=";")
    t = pd.read_csv(trade_path, sep=";")

    p["file_day"] = day
    t["file_day"] = day

    prices_parts.append(p)
    trades_parts.append(t)

prices = pd.concat(prices_parts, ignore_index=True)
trades = pd.concat(trades_parts, ignore_index=True)

# Standardise trade product column name.
if "symbol" in trades.columns and "product" not in trades.columns:
    trades = trades.rename(columns={"symbol": "product"})

# Keep only valid Round 5 algorithmic products.
prices = prices[prices["product"].isin(ALGO_PRODUCTS)].copy()
trades = trades[trades["product"].isin(ALGO_PRODUCTS)].copy()

# Useful global time index across days.
# Assumes timestamp resets each day.
min_day = min(DAYS)
prices["global_ts"] = (prices["file_day"] - min_day) * 1_000_000 + prices["timestamp"]
trades["global_ts"] = (trades["file_day"] - min_day) * 1_000_000 + trades["timestamp"]

prices = prices.sort_values(["product", "global_ts"]).reset_index(drop=True)
trades = trades.sort_values(["product", "global_ts"]).reset_index(drop=True)

# Basic sanity checks.
price_products = sorted(prices["product"].unique())
trade_products = sorted(trades["product"].unique())

missing_in_prices = sorted(set(ALGO_PRODUCTS) - set(price_products))
missing_in_trades = sorted(set(ALGO_PRODUCTS) - set(trade_products))

print("prices shape:", prices.shape)
print("trades shape:", trades.shape)
print()
print("Price products:", price_products)
print("Trade products:", trade_products)
print()
print("Missing in prices:", missing_in_prices)
print("Missing in trades:", missing_in_trades)
print()
print("Price days:", sorted(prices["file_day"].unique()))
print("Trade days:", sorted(trades["file_day"].unique()))
print()
print("Position limits:", POSITION_LIMITS)

display(prices.head())
display(trades.head())

prices shape: (1500000, 19)
trades shape: (35385, 9)

Price products: ['GALAXY_SOUNDS_BLACK_HOLES', 'GALAXY_SOUNDS_DARK_MATTER', 'GALAXY_SOUNDS_PLANETARY_RINGS', 'GALAXY_SOUNDS_SOLAR_FLAMES', 'GALAXY_SOUNDS_SOLAR_WINDS', 'MICROCHIP_CIRCLE', 'MICROCHIP_OVAL', 'MICROCHIP_RECTANGLE', 'MICROCHIP_SQUARE', 'MICROCHIP_TRIANGLE', 'OXYGEN_SHAKE_CHOCOLATE', 'OXYGEN_SHAKE_EVENING_BREATH', 'OXYGEN_SHAKE_GARLIC', 'OXYGEN_SHAKE_MINT', 'OXYGEN_SHAKE_MORNING_BREATH', 'PANEL_1X2', 'PANEL_1X4', 'PANEL_2X2', 'PANEL_2X4', 'PANEL_4X4', 'PEBBLES_L', 'PEBBLES_M', 'PEBBLES_S', 'PEBBLES_XL', 'PEBBLES_XS', 'ROBOT_DISHES', 'ROBOT_IRONING', 'ROBOT_LAUNDRY', 'ROBOT_MOPPING', 'ROBOT_VACUUMING', 'SLEEP_POD_COTTON', 'SLEEP_POD_LAMB_WOOL', 'SLEEP_POD_NYLON', 'SLEEP_POD_POLYESTER', 'SLEEP_POD_SUEDE', 'SNACKPACK_CHOCOLATE', 'SNACKPACK_PISTACHIO', 'SNACKPACK_RASPBERRY', 'SNACKPACK_STRAWBERRY', 'SNACKPACK_VANILLA', 'TRANSLATOR_ASTRO_BLACK', 'TRANSLATOR_ECLIPSE_CHARCOAL', 'TRANSLATOR_GRAPHITE_MIST', 'TRANSLATOR_SPACE_GRAY'

,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss,file_day,global_ts
0,2,0,GALAXY_SOUNDS_BLACK_HOLES,9994,22,9992.0,26.0,NaN,NaN,10006,22,10008.0,26.0,NaN,NaN,10000.0,0.0,2,0
1,2,100,GALAXY_SOUNDS_BLACK_HOLES,10001,18,10000.0,25.0,NaN,NaN,10014,18,10016.0,25.0,NaN,NaN,10007.5,0.0,2,100
2,2,200,GALAXY_SOUNDS_BLACK_HOLES,9996,19,9995.0,31.0,NaN,NaN,10009,19,10011.0,31.0,NaN,NaN,10002.5,0.0,2,200
3,2,300,GALAXY_SOUNDS_BLACK_HOLES,9994,25,9993.0,33.0,NaN,NaN,10007,25,10009.0,33.0,NaN,NaN,10000.5,0.0,2,300
4,2,400,GALAXY_SOUNDS_BLACK_HOLES,9999,14,9997.0,32.0,NaN,NaN,10012,14,10013.0,32.0,NaN,NaN,10005.5,0.0,2,400


,timestamp,buyer,seller,product,currency,price,quantity,file_day,global_ts
0,1700,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9969.0,4,2,1700
1,14500,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9749.0,1,2,14500
2,15100,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9764.0,2,2,15100
3,26500,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9656.0,4,2,26500
4,36400,NaN,NaN,GALAXY_SOUNDS_BLACK_HOLES,XIRECS,9675.0,4,2,36400


In [2]:
import os
import time
import itertools
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

try:
    from IPython.display import display
except Exception:
    display = print

t0 = time.time()
OUTDIR = "analysis_outputs/sleeping_pods_comprehensive"
os.makedirs(OUTDIR, exist_ok=True)

# ============================================================
# 0) CONFIG
# ============================================================

EXPECTED_PODS = [
    "SLEEP_POD_SUEDE",
    "SLEEP_POD_LAMB_WOOL",
    "SLEEP_POD_POLYESTER",
    "SLEEP_POD_NYLON",
    "SLEEP_POD_COTTON",
]

WINDOWS_SCAN = [250, 500, 1000, 2500, 5000]
HORIZONS_SCAN = [100, 250, 500, 1000, 1500, 2500]
THRESHOLDS_SCAN = [1.5, 2.0, 2.5, 3.0]

WINDOWS_STRICT = [250, 500, 1000, 2500, 5000]
ENTRY_ZS = [1.5, 2.0, 2.5, 3.0]
EXIT_ZS = [0.0, 0.25, 0.5, 1.0]
MAX_HOLDS = [250, 500, 1000, 1500, 2500]

TOP_N_STRICT = 60
QTY = 10.0

# ============================================================
# 1) FIND MARKET DATAFRAME AND BUILD POD ARRAYS
# ============================================================

def find_market_df():
    preferred = ["prices_f", "prices", "px", "micro_prices"]
    for name in preferred:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            df = globals()[name]
            if "product" in df.columns and ("bid_price_1" in df.columns) and ("ask_price_1" in df.columns):
                return name, df.copy()
    raise NameError("Could not find a market dataframe. Expected one of prices_f/prices/px/micro_prices.")

market_name, market_df = find_market_df()

day_col = "file_day" if "file_day" in market_df.columns else "day"
mid_col = "mid_price" if "mid_price" in market_df.columns else "mid"

all_products = sorted(market_df["product"].dropna().unique())
found_pods = [p for p in all_products if "SLEEP" in p.upper() and "POD" in p.upper()]

if all(p in all_products for p in EXPECTED_PODS):
    POD_PRODUCTS = EXPECTED_PODS
else:
    POD_PRODUCTS = found_pods

if len(POD_PRODUCTS) != 5:
    raise ValueError(f"Expected 5 sleeping pod products, found {len(POD_PRODUCTS)}: {POD_PRODUCTS}")

POD_IDX = {p: i for i, p in enumerate(POD_PRODUCTS)}

pod_df = market_df[market_df["product"].isin(POD_PRODUCTS)].copy()
days = sorted([int(x) for x in pod_df[day_col].unique()])

MID_DAY, BID_DAY, ASK_DAY, TS_DAY = {}, {}, {}, {}

for d in days:
    ddf = pod_df[pod_df[day_col] == d].copy()
    piv_mid = ddf.pivot_table(index="timestamp", columns="product", values=mid_col, aggfunc="last")
    piv_bid = ddf.pivot_table(index="timestamp", columns="product", values="bid_price_1", aggfunc="last")
    piv_ask = ddf.pivot_table(index="timestamp", columns="product", values="ask_price_1", aggfunc="last")

    piv_mid = piv_mid.reindex(columns=POD_PRODUCTS).sort_index()
    piv_bid = piv_bid.reindex(columns=POD_PRODUCTS).sort_index()
    piv_ask = piv_ask.reindex(columns=POD_PRODUCTS).sort_index()

    # Drop rows where any product is missing
    valid = ~(piv_mid.isna().any(axis=1) | piv_bid.isna().any(axis=1) | piv_ask.isna().any(axis=1))
    piv_mid, piv_bid, piv_ask = piv_mid[valid], piv_bid[valid], piv_ask[valid]

    TS_DAY[d] = piv_mid.index.to_numpy()
    MID_DAY[d] = piv_mid.to_numpy(float)
    BID_DAY[d] = piv_bid.to_numpy(float)
    ASK_DAY[d] = piv_ask.to_numpy(float)

print(f"Market df: {market_name}")
print(f"Sleeping pod products: {POD_PRODUCTS}")
print(f"Days: {days}")
for d in days:
    print(f"Day {d}: mid={MID_DAY[d].shape}, bid={BID_DAY[d].shape}, ask={ASK_DAY[d].shape}")

# ============================================================
# 2) OPTIONAL TRADE FLOW ARRAYS
# ============================================================

FLOW_DAY = None

def build_flow_arrays():
    trade_df = None
    for name in ["micro_trade_q", "day_trades", "trades", "trade_history"]:
        if name in globals() and isinstance(globals()[name], pd.DataFrame):
            df = globals()[name]
            if "product" in df.columns and "timestamp" in df.columns:
                trade_df = df.copy()
                break

    if trade_df is None:
        return None

    dcol = "file_day" if "file_day" in trade_df.columns else ("day" if "day" in trade_df.columns else None)
    if dcol is None:
        return None

    if "signed_qty" in trade_df.columns:
        qty_col = "signed_qty"
    elif "quantity" in trade_df.columns and "inferred_side" in trade_df.columns:
        trade_df["signed_qty"] = trade_df["quantity"] * trade_df["inferred_side"]
        qty_col = "signed_qty"
    else:
        return None

    out = {}
    tdf = trade_df[trade_df["product"].isin(POD_PRODUCTS)].copy()

    for d in days:
        arr = np.zeros_like(MID_DAY[d], dtype=float)
        ts_to_i = {int(ts): i for i, ts in enumerate(TS_DAY[d])}
        sub = tdf[tdf[dcol].astype(int) == int(d)]
        if len(sub) == 0:
            out[d] = arr
            continue

        grouped = sub.groupby(["timestamp", "product"])[qty_col].sum()
        for (ts, prod), val in grouped.items():
            ts = int(ts)
            if ts in ts_to_i and prod in POD_IDX:
                arr[ts_to_i[ts], POD_IDX[prod]] += float(val)
        out[d] = arr

    return out

FLOW_DAY = build_flow_arrays()
print("Flow arrays:", "built" if FLOW_DAY is not None else "not available / skipped")

# ============================================================
# 3) HELPERS
# ============================================================

def safe_log_ratio(a, b):
    return np.log(np.maximum(a, 1e-9) / np.maximum(b, 1e-9))

def rolling_z(x, window):
    s = pd.Series(x)
    minp = max(50, window // 5)
    mean = s.rolling(window, min_periods=minp).mean().shift(1)
    std = s.rolling(window, min_periods=minp).std(ddof=0).shift(1)
    z = (s - mean) / std.replace(0, np.nan)
    return z.to_numpy(float)

def side_from_z(z, mode):
    s = np.sign(z)
    if mode in ("meanrev", "inverse"):
        return -s
    elif mode == "follow":
        return s
    else:
        raise ValueError(f"Unknown mode: {mode}")

def pnl_for_position(pos, entry_i, exit_i, bid, ask):
    pos = np.asarray(pos, dtype=float)
    entry_px = np.where(pos > 0, ask[entry_i], bid[entry_i])
    exit_px = np.where(pos > 0, bid[exit_i], ask[exit_i])
    return float(np.nansum(pos * (exit_px - entry_px)))

def vector_event_pnl(pos_mat, entry_idx, exit_idx, bid, ask):
    # pos_mat shape: events x products
    entry_bid = bid[entry_idx]
    entry_ask = ask[entry_idx]
    exit_bid = bid[exit_idx]
    exit_ask = ask[exit_idx]

    entry_px = np.where(pos_mat > 0, entry_ask, entry_bid)
    exit_px = np.where(pos_mat > 0, exit_bid, exit_ask)
    return np.nansum(pos_mat * (exit_px - entry_px), axis=1)

def avg_group(X, inds):
    return np.mean(X[:, list(inds)], axis=1)

def q_for_groups(A, B):
    q = np.zeros(len(POD_PRODUCTS), dtype=float)
    for i in A:
        q[i] = QTY
    for i in B:
        q[i] = -QTY
    return q

def add_candidate(candidates, name, family, signal_type, note, mode, q_trade, raw_by_day):
    q_trade = np.asarray(q_trade, dtype=float)
    if np.nanmax(np.abs(q_trade)) > QTY:
        raise ValueError(f"{name}: q_trade exceeds per-product qty cap: {q_trade}")
    candidates.append({
        "candidate": name,
        "family": family,
        "signal_type": signal_type,
        "note": note,
        "mode": mode,
        "q_trade": q_trade,
        "raw_by_day": raw_by_day,
    })

def poly_resid_by_day(y_by_day, x_by_day, deg):
    out = {}
    for d in days:
        train_x = np.concatenate([x_by_day[od] for od in days if od != d])
        train_y = np.concatenate([y_by_day[od] for od in days if od != d])
        valid = np.isfinite(train_x) & np.isfinite(train_y)
        if valid.sum() < deg + 5:
            out[d] = np.full_like(x_by_day[d], np.nan, dtype=float)
            continue
        coeff = np.polyfit(train_x[valid], train_y[valid], deg)
        out[d] = y_by_day[d] - np.polyval(coeff, x_by_day[d])
    return out

# ============================================================
# 4) GENERATE COMPREHENSIVE CANDIDATES
# ============================================================

candidates = []

# 4.1 Single-product mean reversion
for p, i in POD_IDX.items():
    raw = {d: MID_DAY[d][:, i].copy() for d in days}
    q = np.zeros(len(POD_PRODUCTS)); q[i] = QTY
    add_candidate(
        candidates,
        f"{p}_own_meanrev",
        "single_product",
        "own_price_z",
        f"{p} own-price rolling mean reversion",
        "meanrev",
        q,
        raw,
    )

# 4.2 Single-product momentum/reversal on own moves
for lag in [50, 100, 250, 500, 1000]:
    for p, i in POD_IDX.items():
        raw = {}
        for d in days:
            x = MID_DAY[d][:, i]
            r = np.full(len(x), np.nan)
            r[lag:] = x[lag:] - x[:-lag]
            raw[d] = r

        q = np.zeros(len(POD_PRODUCTS)); q[i] = QTY

        add_candidate(
            candidates,
            f"{p}_own_momentum_lag{lag}",
            "own_move",
            "momentum",
            f"{p} follows own {lag}-row move",
            "follow",
            q,
            raw,
        )

        add_candidate(
            candidates,
            f"{p}_own_reversal_lag{lag}",
            "own_move",
            "reversal",
            f"{p} reverses own {lag}-row move",
            "inverse",
            q,
            raw,
        )

# 4.3 Pairwise relationships: spread, ratio, logratio
for a, b in itertools.combinations(range(len(POD_PRODUCTS)), 2):
    pa, pb = POD_PRODUCTS[a], POD_PRODUCTS[b]
    q = np.zeros(len(POD_PRODUCTS)); q[a] = QTY; q[b] = -QTY

    raw_spread = {d: MID_DAY[d][:, a] - MID_DAY[d][:, b] for d in days}
    add_candidate(
        candidates,
        f"{pa}_vs_{pb}_spread_mr",
        "pairwise",
        "spread",
        f"{pa} - {pb}",
        "meanrev",
        q,
        raw_spread,
    )

    raw_log = {d: safe_log_ratio(MID_DAY[d][:, a], MID_DAY[d][:, b]) for d in days}
    add_candidate(
        candidates,
        f"{pa}_vs_{pb}_logratio_mr",
        "pairwise",
        "logratio",
        f"log({pa}/{pb})",
        "meanrev",
        q,
        raw_log,
    )

    raw_ratio = {d: MID_DAY[d][:, a] / np.maximum(MID_DAY[d][:, b], 1e-9) for d in days}
    add_candidate(
        candidates,
        f"{pa}_vs_{pb}_ratio_mr",
        "pairwise",
        "ratio",
        f"{pa}/{pb}",
        "meanrev",
        q,
        raw_ratio,
    )

# 4.4 Pairwise nonlinear residuals, out-of-day fitted
for y, x in itertools.permutations(range(len(POD_PRODUCTS)), 2):
    py, px_ = POD_PRODUCTS[y], POD_PRODUCTS[x]
    q = np.zeros(len(POD_PRODUCTS)); q[y] = QTY; q[x] = -QTY

    y_by = {d: MID_DAY[d][:, y] for d in days}
    x_by = {d: MID_DAY[d][:, x] for d in days}

    for deg in [2, 3]:
        raw = poly_resid_by_day(y_by, x_by, deg)
        add_candidate(
            candidates,
            f"{py}_resid_vs_{px_}_poly{deg}_mr",
            "pairwise_nonlinear",
            f"poly{deg}_resid",
            f"{py} residual versus degree-{deg} function of {px_}, fitted out-of-day",
            "meanrev",
            q,
            raw,
        )

# 4.5 Group relationships: every subset versus complement
n = len(POD_PRODUCTS)
all_idx = set(range(n))

for r in range(1, n):
    for A_tuple in itertools.combinations(range(n), r):
        A = set(A_tuple)
        B = all_idx - A
        if not B:
            continue

        # Avoid exact duplicate complements
        mask_A = sum(1 << i for i in A)
        mask_B = sum(1 << i for i in B)
        if mask_A > mask_B:
            continue

        A = tuple(sorted(A))
        B = tuple(sorted(B))
        A_name = "_".join([POD_PRODUCTS[i].replace("SLEEP_POD_", "") for i in A])
        B_name = "_".join([POD_PRODUCTS[i].replace("SLEEP_POD_", "") for i in B])
        q = q_for_groups(A, B)

        raw_log = {
            d: safe_log_ratio(avg_group(MID_DAY[d], A), avg_group(MID_DAY[d], B))
            for d in days
        }
        add_candidate(
            candidates,
            f"group_{A_name}_vs_{B_name}_logratio_mr",
            "group_vs_group",
            "logratio",
            f"log(avg({A_name}) / avg({B_name}))",
            "meanrev",
            q,
            raw_log,
        )

        raw_spread = {
            d: avg_group(MID_DAY[d], A) - avg_group(MID_DAY[d], B)
            for d in days
        }
        add_candidate(
            candidates,
            f"group_{A_name}_vs_{B_name}_spread_mr",
            "group_vs_group",
            "spread",
            f"avg({A_name}) - avg({B_name})",
            "meanrev",
            q,
            raw_spread,
        )

        y_by = {d: avg_group(MID_DAY[d], A) for d in days}
        x_by = {d: avg_group(MID_DAY[d], B) for d in days}
        for deg in [2, 3]:
            raw = poly_resid_by_day(y_by, x_by, deg)
            add_candidate(
                candidates,
                f"group_{A_name}_resid_vs_{B_name}_poly{deg}_mr",
                "group_nonlinear",
                f"poly{deg}_resid",
                f"avg({A_name}) residual versus degree-{deg} function of avg({B_name}), fitted out-of-day",
                "meanrev",
                q,
                raw,
            )

# 4.6 Lead-lag relationships
for lag in [50, 100, 250, 500, 1000]:
    for lead, target in itertools.permutations(range(len(POD_PRODUCTS)), 2):
        lead_p, target_p = POD_PRODUCTS[lead], POD_PRODUCTS[target]
        raw = {}
        for d in days:
            x = MID_DAY[d][:, lead]
            r = np.full(len(x), np.nan)
            r[lag:] = x[lag:] - x[:-lag]
            raw[d] = r

        q = np.zeros(len(POD_PRODUCTS)); q[target] = QTY

        add_candidate(
            candidates,
            f"lead_{lead_p}_to_{target_p}_follow_lag{lag}",
            "leadlag",
            "leadlag_follow",
            f"{lead_p} {lag}-row move predicts {target_p} same direction",
            "follow",
            q,
            raw,
        )

        add_candidate(
            candidates,
            f"lead_{lead_p}_to_{target_p}_inverse_lag{lag}",
            "leadlag",
            "leadlag_inverse",
            f"{lead_p} {lag}-row move predicts {target_p} opposite direction",
            "inverse",
            q,
            raw,
        )

# 4.7 Trade-flow candidates, if trade data exists
if FLOW_DAY is not None:
    for flow_span in [50, 100, 250, 500, 1000, 2500]:
        for p, i in POD_IDX.items():
            raw = {}
            for d in days:
                s = pd.Series(FLOW_DAY[d][:, i])
                raw[d] = s.rolling(flow_span, min_periods=max(5, flow_span // 10)).sum().shift(1).to_numpy(float)

            q = np.zeros(len(POD_PRODUCTS)); q[i] = QTY

            add_candidate(
                candidates,
                f"flow_{p}_follow_span{flow_span}",
                "trade_flow",
                "flow_follow",
                f"{p} follows signed trade-flow imbalance over {flow_span} rows",
                "follow",
                q,
                raw,
            )

            add_candidate(
                candidates,
                f"flow_{p}_inverse_span{flow_span}",
                "trade_flow",
                "flow_inverse",
                f"{p} reverses signed trade-flow imbalance over {flow_span} rows",
                "inverse",
                q,
                raw,
            )

print(f"Candidates generated: {len(candidates)}")

# ============================================================
# 5) BROAD OVERLAPPING EVENT SCAN
# ============================================================

event_rows = []
scan_total = len(candidates) * len(WINDOWS_SCAN) * len(HORIZONS_SCAN) * len(THRESHOLDS_SCAN)
k = 0

for cand in candidates:
    for window in WINDOWS_SCAN:
        z_by_day = {d: rolling_z(cand["raw_by_day"][d], window) for d in days}

        for horizon in HORIZONS_SCAN:
            for th in THRESHOLDS_SCAN:
                k += 1
                if k == 1 or k % 500 == 0:
                    print(f"[{time.time()-t0:7.2f}s] scan {k}/{scan_total}: {cand['candidate']}, w={window}, h={horizon}, th={th}")

                day_pnls = []
                day_hits = []
                day_events = []

                for d in days:
                    z = z_by_day[d]
                    nrows = len(z)
                    valid = np.isfinite(z)
                    idx = np.where(valid & (np.abs(z) >= th))[0]
                    idx = idx[idx + horizon < nrows]

                    if len(idx) == 0:
                        day_pnls.append(0.0)
                        day_hits.append(np.nan)
                        day_events.append(0)
                        continue

                    sides = side_from_z(z[idx], cand["mode"])
                    pos_mat = sides[:, None] * cand["q_trade"][None, :]

                    pnls = vector_event_pnl(
                        pos_mat,
                        idx,
                        idx + horizon,
                        BID_DAY[d],
                        ASK_DAY[d],
                    )

                    day_pnls.append(float(np.nansum(pnls)))
                    day_hits.append(float(np.mean(pnls > 0)))
                    day_events.append(int(len(pnls)))

                total_pnl = float(np.sum(day_pnls))
                min_day_pnl = float(np.min(day_pnls))
                max_day_pnl = float(np.max(day_pnls))
                total_events = int(np.sum(day_events))
                active_days = int(np.sum(np.array(day_events) > 0))
                positive_days = int(np.sum(np.array(day_pnls) > 0))
                mean_hit = float(np.nanmean(day_hits)) if np.isfinite(day_hits).any() else np.nan

                if total_events == 0:
                    continue

                one_day_dependency = max_day_pnl / total_pnl if total_pnl > 0 else np.inf
                robust_pass = active_days == len(days) and positive_days == len(days) and min_day_pnl > 0

                robust_score = (
                    total_pnl
                    + 3.0 * min_day_pnl
                    + 5000.0 * (mean_hit if np.isfinite(mean_hit) else 0)
                    - 0.25 * max_day_pnl
                    - 1000.0 * max(0, one_day_dependency - 0.75 if np.isfinite(one_day_dependency) else 10)
                )

                event_rows.append({
                    "candidate": cand["candidate"],
                    "family": cand["family"],
                    "signal_type": cand["signal_type"],
                    "note": cand["note"],
                    "mode": cand["mode"],
                    "window": window,
                    "horizon": horizon,
                    "threshold": th,
                    "days": len(days),
                    "active_days": active_days,
                    "positive_days": positive_days,
                    "total_events": total_events,
                    "total_pnl": total_pnl,
                    "mean_day_pnl": total_pnl / len(days),
                    "min_day_pnl": min_day_pnl,
                    "max_day_pnl": max_day_pnl,
                    "mean_hit_rate": mean_hit,
                    "one_day_dependency": one_day_dependency,
                    "robust_pass": robust_pass,
                    "robust_score": robust_score,
                    "q_trade": cand["q_trade"].tolist(),
                })

EVENT_SCAN = pd.DataFrame(event_rows)
EVENT_SCAN = EVENT_SCAN.sort_values(
    ["robust_pass", "robust_score", "total_pnl"],
    ascending=[False, False, False],
).reset_index(drop=True)

EVENT_SCAN.to_csv(f"{OUTDIR}/pod_event_scan.csv", index=False)

print("\nTop POD_EVENT_SCAN:")
display(EVENT_SCAN.head(50))

# ============================================================
# 6) STRICT POSITION BACKTEST ON TOP CANDIDATES
# ============================================================

top_candidate_names = (
    EVENT_SCAN
    .drop_duplicates("candidate")
    .head(TOP_N_STRICT)["candidate"]
    .tolist()
)

strict_candidates = [c for c in candidates if c["candidate"] in set(top_candidate_names)]

print(f"\nStrict candidates: {len(strict_candidates)}")

def strict_backtest_day(cand, d, window, entry_z, exit_z, max_hold):
    raw = cand["raw_by_day"][d]
    z = rolling_z(raw, window)

    bid = BID_DAY[d]
    ask = ASK_DAY[d]
    ts = TS_DAY[d]
    nrows = len(z)

    trades = []
    i = 0

    while i < nrows - 1:
        if not np.isfinite(z[i]) or abs(z[i]) < entry_z:
            i += 1
            continue

        entry_sign = np.sign(z[i])
        side = side_from_z(z[i], cand["mode"])
        if side == 0 or not np.isfinite(side):
            i += 1
            continue

        pos = side * cand["q_trade"]
        if np.all(pos == 0):
            i += 1
            continue

        entry_i = i
        j = i + 1

        while j < nrows - 1:
            held = j - entry_i

            exit_signal = False
            if np.isfinite(z[j]):
                # exit once signal has reverted enough relative to entry direction
                if entry_sign * z[j] <= exit_z:
                    exit_signal = True

            exit_hold = held >= max_hold

            if exit_signal or exit_hold:
                break

            j += 1

        exit_i = min(j, nrows - 1)
        pnl = pnl_for_position(pos, entry_i, exit_i, bid, ask)

        trades.append({
            "candidate": cand["candidate"],
            "family": cand["family"],
            "signal_type": cand["signal_type"],
            "note": cand["note"],
            "mode": cand["mode"],
            "day": d,
            "window": window,
            "entry_z": entry_z,
            "exit_z": exit_z,
            "max_hold": max_hold,
            "entry_idx": entry_i,
            "exit_idx": exit_i,
            "entry_ts": int(ts[entry_i]),
            "exit_ts": int(ts[exit_i]),
            "side": float(side),
            "entry_z_seen": float(z[entry_i]),
            "exit_z_seen": float(z[exit_i]) if np.isfinite(z[exit_i]) else np.nan,
            "hold": int(exit_i - entry_i),
            "exec_pnl": float(pnl),
            "q_trade": cand["q_trade"].tolist(),
            "position": pos.tolist(),
        })

        i = exit_i + 1

    return trades

strict_day_rows = []
strict_trade_rows = []

strict_total = len(strict_candidates) * len(WINDOWS_STRICT) * len(ENTRY_ZS) * len(EXIT_ZS) * len(MAX_HOLDS)
k = 0

for cand in strict_candidates:
    for window in WINDOWS_STRICT:
        for entry_z in ENTRY_ZS:
            for exit_z in EXIT_ZS:
                for max_hold in MAX_HOLDS:
                    k += 1
                    if k == 1 or k % 250 == 0:
                        print(f"[{time.time()-t0:7.2f}s] strict {k}/{strict_total}: {cand['candidate']}, w={window}, entry={entry_z}, exit={exit_z}, hold={max_hold}")

                    for d in days:
                        trades = strict_backtest_day(cand, d, window, entry_z, exit_z, max_hold)

                        day_pnl = float(sum(t["exec_pnl"] for t in trades))
                        trade_count = len(trades)
                        hit_rate = float(np.mean([t["exec_pnl"] > 0 for t in trades])) if trades else np.nan
                        avg_trade_pnl = float(np.mean([t["exec_pnl"] for t in trades])) if trades else 0.0
                        median_trade_pnl = float(np.median([t["exec_pnl"] for t in trades])) if trades else 0.0

                        strict_day_rows.append({
                            "candidate": cand["candidate"],
                            "family": cand["family"],
                            "signal_type": cand["signal_type"],
                            "note": cand["note"],
                            "mode": cand["mode"],
                            "day": d,
                            "window": window,
                            "entry_z": entry_z,
                            "exit_z": exit_z,
                            "max_hold": max_hold,
                            "trade_count": trade_count,
                            "day_pnl": day_pnl,
                            "hit_rate": hit_rate,
                            "avg_trade_pnl": avg_trade_pnl,
                            "median_trade_pnl": median_trade_pnl,
                            "q_trade": cand["q_trade"].tolist(),
                        })

                        strict_trade_rows.extend(trades)

POD_STRICT_DAY_RESULTS = pd.DataFrame(strict_day_rows)
POD_STRICT_TRADE_LOG = pd.DataFrame(strict_trade_rows)

def summarise_strict(day_df):
    gcols = ["candidate", "family", "signal_type", "note", "mode", "window", "entry_z", "exit_z", "max_hold"]

    rows = []
    for keys, g in day_df.groupby(gcols, dropna=False):
        day_pnls = g["day_pnl"].to_numpy(float)
        trade_counts = g["trade_count"].to_numpy(int)

        total_pnl = float(day_pnls.sum())
        min_day_pnl = float(day_pnls.min())
        max_day_pnl = float(day_pnls.max())
        active_days = int((trade_counts > 0).sum())
        positive_days = int((day_pnls > 0).sum())
        total_trades = int(trade_counts.sum())
        mean_hit = float(g["hit_rate"].mean(skipna=True)) if g["hit_rate"].notna().any() else np.nan
        min_hit = float(g["hit_rate"].min(skipna=True)) if g["hit_rate"].notna().any() else np.nan
        pnl_per_trade = total_pnl / total_trades if total_trades else np.nan
        one_day_dependency = max_day_pnl / total_pnl if total_pnl > 0 else np.inf

        robust_pass = (
            active_days == len(days)
            and positive_days == len(days)
            and min_day_pnl > 0
            and total_trades >= 3
        )

        robust_score = (
            total_pnl
            + 2.5 * min_day_pnl
            + 3000.0 * (mean_hit if np.isfinite(mean_hit) else 0)
            - 0.25 * max_day_pnl
            - 2000.0 * max(0, one_day_dependency - 0.75 if np.isfinite(one_day_dependency) else 10)
        )

        row = dict(zip(gcols, keys if isinstance(keys, tuple) else (keys,)))
        row.update({
            "days": len(g),
            "active_days": active_days,
            "positive_days": positive_days,
            "total_pnl": total_pnl,
            "mean_day_pnl": total_pnl / len(days),
            "min_day_pnl": min_day_pnl,
            "max_day_pnl": max_day_pnl,
            "total_trades": total_trades,
            "mean_hit_rate": mean_hit,
            "min_hit_rate": min_hit,
            "pnl_per_trade": pnl_per_trade,
            "one_day_dependency": one_day_dependency,
            "robust_pass": robust_pass,
            "robust_score": robust_score,
        })
        rows.append(row)

    return pd.DataFrame(rows)

POD_STRICT_SUMMARY = summarise_strict(POD_STRICT_DAY_RESULTS)
POD_STRICT_SUMMARY = POD_STRICT_SUMMARY.sort_values(
    ["robust_pass", "robust_score", "total_pnl"],
    ascending=[False, False, False],
).reset_index(drop=True)

POD_STRICT_ROBUST_ONLY = POD_STRICT_SUMMARY[POD_STRICT_SUMMARY["robust_pass"]].copy()

POD_STRICT_DAY_RESULTS.to_csv(f"{OUTDIR}/pod_strict_day_results.csv", index=False)
POD_STRICT_TRADE_LOG.to_csv(f"{OUTDIR}/pod_strict_trade_log.csv", index=False)
POD_STRICT_SUMMARY.to_csv(f"{OUTDIR}/pod_strict_summary.csv", index=False)
POD_STRICT_ROBUST_ONLY.to_csv(f"{OUTDIR}/pod_strict_robust_only.csv", index=False)

print("\nTop POD_STRICT_SUMMARY:")
display(POD_STRICT_SUMMARY.head(50))

print("\nPOD_STRICT_ROBUST_ONLY:")
display(POD_STRICT_ROBUST_ONLY.head(50))

# ============================================================
# 7) BEST CONFIG BREAKDOWNS
# ============================================================

if len(POD_STRICT_SUMMARY):
    best = POD_STRICT_SUMMARY.iloc[0]
    best_filter = (
        (POD_STRICT_DAY_RESULTS["candidate"] == best["candidate"])
        & (POD_STRICT_DAY_RESULTS["window"] == best["window"])
        & (POD_STRICT_DAY_RESULTS["entry_z"] == best["entry_z"])
        & (POD_STRICT_DAY_RESULTS["exit_z"] == best["exit_z"])
        & (POD_STRICT_DAY_RESULTS["max_hold"] == best["max_hold"])
    )

    BEST_POD_DAY_BREAKDOWN = POD_STRICT_DAY_RESULTS[best_filter].copy()

    if len(POD_STRICT_TRADE_LOG):
        best_tr_filter = (
            (POD_STRICT_TRADE_LOG["candidate"] == best["candidate"])
            & (POD_STRICT_TRADE_LOG["window"] == best["window"])
            & (POD_STRICT_TRADE_LOG["entry_z"] == best["entry_z"])
            & (POD_STRICT_TRADE_LOG["exit_z"] == best["exit_z"])
            & (POD_STRICT_TRADE_LOG["max_hold"] == best["max_hold"])
        )
        BEST_POD_TRADES = POD_STRICT_TRADE_LOG[best_tr_filter].copy()
    else:
        BEST_POD_TRADES = pd.DataFrame()

    print("\nBest strict config:")
    display(best.to_frame().T)

    print("\nBest config day breakdown:")
    display(BEST_POD_DAY_BREAKDOWN)

    print("\nBest config trades:")
    display(BEST_POD_TRADES.head(100))

    BEST_POD_DAY_BREAKDOWN.to_csv(f"{OUTDIR}/best_pod_day_breakdown.csv", index=False)
    BEST_POD_TRADES.to_csv(f"{OUTDIR}/best_pod_trades.csv", index=False)

print(f"\nSaved outputs to: {OUTDIR}")
print(f"Runtime: {time.time() - t0:.2f} seconds")

Market df: prices
Sleeping pod products: ['SLEEP_POD_SUEDE', 'SLEEP_POD_LAMB_WOOL', 'SLEEP_POD_POLYESTER', 'SLEEP_POD_NYLON', 'SLEEP_POD_COTTON']
Days: [2, 3, 4]
Day 2: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Day 3: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Day 4: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Flow arrays: not available / skipped
Candidates generated: 385
[   0.44s] scan 1/46200: SLEEP_POD_SUEDE_own_meanrev, w=250, h=100, th=1.5
[   0.61s] scan 500/46200: SLEEP_POD_COTTON_own_meanrev, w=250, h=1500, th=3.0
[   0.74s] scan 1000/46200: SLEEP_POD_LAMB_WOOL_own_reversal_lag50, w=500, h=1000, th=3.0
[   0.87s] scan 1500/46200: SLEEP_POD_NYLON_own_reversal_lag50, w=1000, h=500, th=3.0
[   1.00s] scan 2000/46200: SLEEP_POD_SUEDE_own_reversal_lag100, w=2500, h=250, th=3.0
[   1.13s] scan 2500/46200: SLEEP_POD_POLYESTER_own_reversal_lag100, w=5000, h=100, th=3.0
[   1.25s] scan 3000/46200: SLEEP_POD_COTTON_own_reversal_lag100, w=5000, h=2500, th=3.0
[   1.39s] scan 

,candidate,family,signal_type,note,mode,window,horizon,threshold,days,active_days,...,total_events,total_pnl,mean_day_pnl,min_day_pnl,max_day_pnl,mean_hit_rate,one_day_dependency,robust_pass,robust_score,q_trade
0,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,spread,avg(POLYESTER_NYLON) - avg(SUEDE_LAMB_WOOL_COT...,meanrev,5000,2500,1.5,3,3,...,4649,49509870.0,1.650329e+07,10516950.0,22736210.0,0.767663,0.459226,True,7.538051e+07,"[-10.0, -10.0, 10.0, 10.0, -10.0]"
1,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,5000,2500,1.5,3,3,...,4521,48351990.0,1.611733e+07,10565420.0,22244750.0,0.764380,0.460059,True,7.449088e+07,"[-10.0, -10.0, 10.0, 10.0, -10.0]"
2,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,spread,avg(POLYESTER_NYLON) - avg(SUEDE_LAMB_WOOL_COT...,meanrev,5000,1500,1.5,3,3,...,5253,43044340.0,1.434811e+07,11416070.0,18996210.0,0.905376,0.441317,True,7.254802e+07,"[-10.0, -10.0, 10.0, 10.0, -10.0]"
3,group_POLYESTER_NYLON_resid_vs_SUEDE_LAMB_WOOL...,group_nonlinear,poly2_resid,avg(POLYESTER_NYLON) residual versus degree-2 ...,meanrev,5000,2500,1.5,3,3,...,4748,50746470.0,1.691549e+07,9233010.0,25732380.0,0.759671,0.507077,True,7.201620e+07,"[-10.0, -10.0, 10.0, 10.0, -10.0]"
4,group_SUEDE_LAMB_WOOL_resid_vs_POLYESTER_NYLON...,group_nonlinear,poly2_resid,avg(SUEDE_LAMB_WOOL) residual versus degree-2 ...,meanrev,5000,2500,1.5,3,3,...,6165,50784870.0,1.692829e+07,9356330.0,28275610.0,0.882810,0.556772,True,7.178937e+07,"[10.0, 10.0, -10.0, -10.0, -10.0]"
5,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,5000,1500,1.5,3,3,...,5147,42186550.0,1.406218e+07,11396320.0,18672940.0,0.901723,0.442628,True,7.171178e+07,"[-10.0, -10.0, 10.0, 10.0, -10.0]"
6,group_SUEDE_LAMB_WOOL_vs_POLYESTER_NYLON_COTTO...,group_vs_group,spread,avg(SUEDE_LAMB_WOOL) - avg(POLYESTER_NYLON_COT...,meanrev,5000,2500,1.5,3,3,...,5423,49195130.0,1.639838e+07,9326900.0,30440080.0,0.899044,0.618762,True,6.957031e+07,"[10.0, 10.0, -10.0, -10.0, -10.0]"
7,group_POLYESTER_vs_SUEDE_LAMB_WOOL_NYLON_COTTO...,group_vs_group,spread,avg(POLYESTER) - avg(SUEDE_LAMB_WOOL_NYLON_COT...,meanrev,5000,2500,1.5,3,3,...,5560,38514490.0,1.283816e+07,11300810.0,15707220.0,0.736022,0.407826,True,6.849380e+07,"[-10.0, -10.0, 10.0, -10.0, -10.0]"
8,group_SUEDE_LAMB_WOOL_vs_POLYESTER_NYLON_COTTO...,group_vs_group,logratio,log(avg(SUEDE_LAMB_WOOL) / avg(POLYESTER_NYLON...,meanrev,5000,2500,1.5,3,3,...,5352,48213050.0,1.607102e+07,9197380.0,29625970.0,0.897866,0.614480,True,6.840319e+07,"[10.0, 10.0, -10.0, -10.0, -10.0]"
9,group_POLYESTER_vs_SUEDE_LAMB_WOOL_NYLON_COTTO...,group_vs_group,logratio,log(avg(POLYESTER) / avg(SUEDE_LAMB_WOOL_NYLON...,meanrev,5000,2500,1.5,3,3,...,5728,38290000.0,1.276333e+07,11070990.0,15707620.0,0.723616,0.410228,True,6.757968e+07,"[-10.0, -10.0, 10.0, -10.0, -10.0]"



Strict candidates: 60
[  14.31s] strict 1/24000: SLEEP_POD_SUEDE_own_meanrev, w=250, entry=1.5, exit=0.0, hold=250
[  17.97s] strict 250/24000: SLEEP_POD_SUEDE_own_meanrev, w=2500, entry=1.5, exit=0.25, hold=2500
[  21.47s] strict 500/24000: SLEEP_POD_LAMB_WOOL_own_meanrev, w=500, entry=1.5, exit=1.0, hold=2500
[  24.89s] strict 750/24000: SLEEP_POD_LAMB_WOOL_own_meanrev, w=5000, entry=2.0, exit=0.25, hold=2500
[  28.45s] strict 1000/24000: SLEEP_POD_LAMB_WOOL_own_momentum_lag500, w=1000, entry=2.0, exit=1.0, hold=2500
[  31.95s] strict 1250/24000: SLEEP_POD_COTTON_own_reversal_lag500, w=250, entry=2.5, exit=0.25, hold=2500
[  35.37s] strict 1500/24000: SLEEP_POD_COTTON_own_reversal_lag500, w=2500, entry=2.5, exit=1.0, hold=2500
[  38.77s] strict 1750/24000: SLEEP_POD_LAMB_WOOL_own_momentum_lag1000, w=500, entry=3.0, exit=0.25, hold=2500
[  42.05s] strict 2000/24000: SLEEP_POD_LAMB_WOOL_own_momentum_lag1000, w=5000, entry=3.0, exit=1.0, hold=2500
[  45.73s] strict 2250/24000: SLEEP_PO

,candidate,family,signal_type,note,mode,window,entry_z,exit_z,max_hold,days,...,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
0,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,5000,1.5,0.25,2500,3,...,23693.333333,20670.0,25310.0,23,0.748148,0.666667,3090.434783,0.356078,True,118671.944444
1,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,5000,1.5,0.25,1500,3,...,23136.666667,19890.0,25100.0,24,0.759259,0.700000,2892.083333,0.361619,True,115137.777778
2,group_POLYESTER_NYLON_resid_vs_SUEDE_LAMB_WOOL...,group_nonlinear,poly3_resid,avg(POLYESTER_NYLON) residual versus degree-3 ...,meanrev,2500,2.5,0.00,2500,3,...,26136.666667,16950.0,33190.0,15,0.866667,0.800000,5227.333333,0.423288,True,115087.500000
3,group_POLYESTER_NYLON_resid_vs_SUEDE_LAMB_WOOL...,group_nonlinear,poly3_resid,avg(POLYESTER_NYLON) residual versus degree-3 ...,meanrev,2500,2.0,0.00,2500,3,...,25860.000000,15750.0,39860.0,18,0.830159,0.800000,4310.000000,0.513792,True,109480.476190
4,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,spread,avg(POLYESTER_NYLON) - avg(SUEDE_LAMB_WOOL_COT...,meanrev,5000,1.5,0.25,2500,3,...,22083.333333,17610.0,24390.0,23,0.783333,0.750000,2880.434783,0.368151,True,106527.500000
5,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,spread,avg(POLYESTER_NYLON) - avg(SUEDE_LAMB_WOOL_COT...,meanrev,5000,1.5,0.25,1500,3,...,21520.000000,17610.0,23640.0,24,0.789394,0.750000,2690.000000,0.366171,True,105043.181818
6,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,spread,avg(POLYESTER_NYLON) - avg(SUEDE_LAMB_WOOL_COT...,meanrev,5000,1.5,0.00,2500,3,...,22850.000000,16790.0,32430.0,20,0.869048,0.750000,3427.500000,0.473085,True,105024.642857
7,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,spread,avg(POLYESTER_NYLON) - avg(SUEDE_LAMB_WOOL_COT...,meanrev,5000,2.0,0.25,2500,3,...,19173.333333,18150.0,20020.0,14,0.944444,0.833333,4108.571429,0.348053,True,100723.333333
8,group_SUEDE_POLYESTER_NYLON_resid_vs_LAMB_WOOL...,group_nonlinear,poly3_resid,avg(SUEDE_POLYESTER_NYLON) residual versus deg...,meanrev,1000,1.5,0.00,2500,3,...,19880.000000,17880.0,23590.0,54,0.732407,0.625000,1104.444444,0.395540,True,100639.722222
9,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,spread,avg(POLYESTER_NYLON) - avg(SUEDE_LAMB_WOOL_COT...,meanrev,5000,1.5,0.00,1500,3,...,21626.666667,16040.0,29510.0,21,0.878307,0.777778,3089.523810,0.454840,True,100237.420635



POD_STRICT_ROBUST_ONLY:


...


Best config day breakdown:


,candidate,family,signal_type,note,mode,day,window,entry_z,exit_z,max_hold,trade_count,day_pnl,hit_rate,avg_trade_pnl,median_trade_pnl,q_trade
51387,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,2,5000,1.5,0.25,2500,5,25310.0,0.800000,5062.000000,5990.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]"
51388,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,3,5000,1.5,0.25,2500,9,25100.0,0.777778,2788.888889,2560.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]"
51389,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,4,5000,1.5,0.25,2500,9,20670.0,0.666667,2296.666667,3010.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]"



Best config trades:


,candidate,family,signal_type,note,mode,day,window,entry_z,exit_z,max_hold,...,exit_idx,entry_ts,exit_ts,side,entry_z_seen,exit_z_seen,hold,exec_pnl,q_trade,position
1005567,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,2,5000,1.5,0.25,2500,...,1778,100000,177800,1.0,-1.609221,-0.226735,778,-1040.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]","[-10.0, -10.0, 10.0, 10.0, -10.0]"
1005568,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,2,5000,1.5,0.25,2500,...,2590,209900,259000,-1.0,1.516277,0.230399,491,5990.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]","[10.0, 10.0, -10.0, -10.0, 10.0]"
1005569,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,2,5000,1.5,0.25,2500,...,4399,287700,439900,1.0,-1.519973,-0.222000,1522,5390.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]","[-10.0, -10.0, 10.0, 10.0, -10.0]"
1005570,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,2,5000,1.5,0.25,2500,...,6803,565400,680300,-1.0,1.524761,0.209612,1149,8590.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]","[10.0, 10.0, -10.0, -10.0, 10.0]"
1005571,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,2,5000,1.5,0.25,2500,...,9225,879300,922500,1.0,-1.503264,-0.138021,432,6380.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]","[-10.0, -10.0, 10.0, 10.0, -10.0]"
1005572,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,3,5000,1.5,0.25,2500,...,1157,103100,115700,1.0,-1.801334,-0.199210,126,2110.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]","[-10.0, -10.0, 10.0, 10.0, -10.0]"
1005573,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,3,5000,1.5,0.25,2500,...,1275,124700,127500,1.0,-1.515565,-0.129002,28,2560.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]","[-10.0, -10.0, 10.0, 10.0, -10.0]"
1005574,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,3,5000,1.5,0.25,2500,...,2857,144400,285700,-1.0,1.620962,0.220867,1413,-200.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]","[10.0, 10.0, -10.0, -10.0, 10.0]"
1005575,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,3,5000,1.5,0.25,2500,...,4023,350000,402300,1.0,-1.512431,-0.226212,523,1610.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]","[-10.0, -10.0, 10.0, 10.0, -10.0]"
1005576,group_POLYESTER_NYLON_vs_SUEDE_LAMB_WOOL_COTTO...,group_vs_group,logratio,log(avg(POLYESTER_NYLON) / avg(SUEDE_LAMB_WOOL...,meanrev,3,5000,1.5,0.25,2500,...,4693,432700,469300,1.0,-1.514391,-0.242754,366,6770.0,"[-10.0, -10.0, 10.0, 10.0, -10.0]","[-10.0, -10.0, 10.0, 10.0, -10.0]"



Saved outputs to: analysis_outputs/sleeping_pods_comprehensive
Runtime: 365.56 seconds


In [3]:
import numpy as np
import pandas as pd
import time
from itertools import product

# ============================================================
# SLEEPING PODS CONFIRMATION / ROBUSTNESS TEST
# Tests:
# - current raw basket
# - net-neutral basket variants
# - scaled baskets
# - flipped sign sanity check
# - spread / logratio / ratio signal forms
# - nearby parameter grid
# ============================================================

POD_PRODUCTS = [
    "SLEEP_POD_SUEDE",
    "SLEEP_POD_LAMB_WOOL",
    "SLEEP_POD_POLYESTER",
    "SLEEP_POD_NYLON",
    "SLEEP_POD_COTTON",
]

OUTDIR = "analysis_outputs/sleeping_pods_confirm"
import os
os.makedirs(OUTDIR, exist_ok=True)

# ----------------------------
# Build pod arrays from prices
# ----------------------------

def build_pod_arrays_from_prices(prices_df, products=POD_PRODUCTS):
    day_col = "file_day" if "file_day" in prices_df.columns else "day"
    needed_cols = [day_col, "timestamp", "product", "bid_price_1", "ask_price_1", "mid_price"]
    missing = [c for c in needed_cols if c not in prices_df.columns]
    if missing:
        raise ValueError(f"prices_df missing columns: {missing}")

    df = prices_df[prices_df["product"].isin(products)].copy()
    days = sorted(df[day_col].unique())

    mid_by_day, bid_by_day, ask_by_day = {}, {}, {}

    for d in days:
        ddf = df[df[day_col] == d].copy()

        mid_piv = ddf.pivot(index="timestamp", columns="product", values="mid_price").sort_index()
        bid_piv = ddf.pivot(index="timestamp", columns="product", values="bid_price_1").sort_index()
        ask_piv = ddf.pivot(index="timestamp", columns="product", values="ask_price_1").sort_index()

        mid_piv = mid_piv[products]
        bid_piv = bid_piv[products]
        ask_piv = ask_piv[products]

        if mid_piv.isna().any().any() or bid_piv.isna().any().any() or ask_piv.isna().any().any():
            raise ValueError(f"NaNs found in day {d} pivot arrays.")

        mid_by_day[int(d)] = mid_piv.to_numpy(dtype=float)
        bid_by_day[int(d)] = bid_piv.to_numpy(dtype=float)
        ask_by_day[int(d)] = ask_piv.to_numpy(dtype=float)

    return days, mid_by_day, bid_by_day, ask_by_day


if "mid_by_day" not in globals() or "bid_by_day" not in globals() or "ask_by_day" not in globals():
    if "prices" not in globals():
        raise NameError("Need either mid_by_day/bid_by_day/ask_by_day or prices dataframe.")
    days, mid_by_day, bid_by_day, ask_by_day = build_pod_arrays_from_prices(prices)
else:
    days = sorted([int(d) for d in mid_by_day.keys()])

print("Days:", days)
for d in days:
    print(f"Day {d}: mid={mid_by_day[d].shape}, bid={bid_by_day[d].shape}, ask={ask_by_day[d].shape}")


# ----------------------------
# Signal definitions
# Product order:
# [SUEDE, LAMB_WOOL, POLYESTER, NYLON, COTTON]
# ----------------------------

def pod_group_values(mid):
    suede = mid[:, 0]
    lamb = mid[:, 1]
    polyester = mid[:, 2]
    nylon = mid[:, 3]
    cotton = mid[:, 4]

    pn = (polyester + nylon) / 2.0
    slc = (suede + lamb + cotton) / 3.0
    return pn, slc


def signal_pn_vs_slc_logratio(mid):
    pn, slc = pod_group_values(mid)
    return np.log(pn / slc)


def signal_pn_vs_slc_spread(mid):
    pn, slc = pod_group_values(mid)
    return pn - slc


def signal_pn_vs_slc_ratio(mid):
    pn, slc = pod_group_values(mid)
    return pn / slc


SIGNALS = {
    "pn_vs_slc_logratio": signal_pn_vs_slc_logratio,
    "pn_vs_slc_spread": signal_pn_vs_slc_spread,
    "pn_vs_slc_ratio": signal_pn_vs_slc_ratio,
}


# ----------------------------
# Rolling z-score
# ----------------------------

def rolling_zscore(x, window):
    s = pd.Series(x)
    mu = s.rolling(window=window, min_periods=window).mean()
    sd = s.rolling(window=window, min_periods=window).std(ddof=0)
    z = (s - mu) / sd
    return z.to_numpy(dtype=float)


# ----------------------------
# Execution PnL
# q is signed position vector.
# Positive q = long, entered at ask, exited at bid.
# Negative q = short, entered at bid, exited at ask.
# ----------------------------

def exec_pnl_for_position(q, bid_entry, ask_entry, bid_exit, ask_exit):
    q = np.asarray(q, dtype=float)
    entry_px = np.where(q > 0, ask_entry, bid_entry)
    exit_px = np.where(q > 0, bid_exit, ask_exit)
    return float(np.sum(q * (exit_px - entry_px)))


def strict_backtest_one_day(mid, bid, ask, signal_func, q_base, window, entry_z, exit_z, max_hold):
    sig = signal_func(mid)
    z = rolling_zscore(sig, window)

    trades = []
    n = len(sig)
    i = 0

    while i < n:
        zi = z[i]

        if not np.isfinite(zi):
            i += 1
            continue

        side = 0

        # Mean reversion:
        # low signal => use q_base
        # high signal => flip q_base
        if zi <= -entry_z:
            side = 1
        elif zi >= entry_z:
            side = -1

        if side == 0:
            i += 1
            continue

        q = side * np.asarray(q_base, dtype=float)
        entry_idx = i

        exit_idx = None
        max_exit = min(n - 1, entry_idx + max_hold)

        j = entry_idx + 1
        while j <= max_exit:
            zj = z[j]

            if np.isfinite(zj):
                if side == 1 and zj >= -exit_z:
                    exit_idx = j
                    break
                if side == -1 and zj <= exit_z:
                    exit_idx = j
                    break

            j += 1

        if exit_idx is None:
            exit_idx = max_exit

        pnl = exec_pnl_for_position(
            q=q,
            bid_entry=bid[entry_idx],
            ask_entry=ask[entry_idx],
            bid_exit=bid[exit_idx],
            ask_exit=ask[exit_idx],
        )

        trades.append({
            "entry_idx": entry_idx,
            "exit_idx": exit_idx,
            "entry_ts": entry_idx * 100,
            "exit_ts": exit_idx * 100,
            "side": side,
            "entry_z_seen": zi,
            "exit_z_seen": z[exit_idx],
            "hold": exit_idx - entry_idx,
            "exec_pnl": pnl,
            "q_trade": list(q_base),
            "position": list(q),
        })

        i = exit_idx + 1

    return trades


def summarize_day_trades(trades):
    if not trades:
        return {
            "trade_count": 0,
            "day_pnl": 0.0,
            "hit_rate": np.nan,
            "avg_trade_pnl": np.nan,
            "median_trade_pnl": np.nan,
        }

    pnls = np.array([t["exec_pnl"] for t in trades], dtype=float)

    return {
        "trade_count": len(trades),
        "day_pnl": float(np.sum(pnls)),
        "hit_rate": float(np.mean(pnls > 0)),
        "avg_trade_pnl": float(np.mean(pnls)),
        "median_trade_pnl": float(np.median(pnls)),
    }


# ----------------------------
# q variants
# Current best q was:
# [-10, -10, +10, +10, -10]
#
# Product order:
# [SUEDE, LAMB_WOOL, POLYESTER, NYLON, COTTON]
# ----------------------------

Q_VARIANTS = {
    # Current discovered best
    "raw_current_10": [-10, -10, +10, +10, -10],

    # Same structure, lower/higher size
    "raw_current_5": [-5, -5, +5, +5, -5],
    "raw_current_15": [-15, -15, +15, +15, -15],

    # Cleaner group-size-neutral variants:
    # 2-product group gets larger per-leg weight than 3-product group.
    "neutral_6_9": [-6, -6, +9, +9, -6],
    "neutral_8_12": [-8, -8, +12, +12, -8],
    "neutral_10_15": [-10, -10, +15, +15, -10],
    "neutral_12_18": [-12, -12, +18, +18, -12],

    # Near-neutral integer approximation with PN capped at 10
    "near_neutral_cap10": [-7, -7, +10, +10, -7],

    # Test whether COTTON is actually helpful as hedge leg
    "no_cotton_10_15": [-10, -10, +15, +15, 0],

    # Explicit sign sanity checks
    "FLIPPED_raw_current_10": [+10, +10, -10, -10, +10],
    "FLIPPED_neutral_10_15": [+10, +10, -15, -15, +10],
}


# ----------------------------
# Parameter grid
# Keep this local-ish around the best config first.
# ----------------------------

WINDOWS = [2500, 4000, 5000, 6000, 7500]
ENTRY_ZS = [1.25, 1.5, 1.75, 2.0, 2.5]
EXIT_ZS = [0.0, 0.25, 0.5, 0.75]
MAX_HOLDS = [1000, 1500, 2500, 4000]


# ----------------------------
# Run test
# ----------------------------

summary_rows = []
day_rows = []
trade_rows = []

configs = list(product(SIGNALS.items(), Q_VARIANTS.items(), WINDOWS, ENTRY_ZS, EXIT_ZS, MAX_HOLDS))
print("Total configs:", len(configs))

t0 = time.time()

for ci, ((signal_name, signal_func), (q_name, q), window, entry_z, exit_z, max_hold) in enumerate(configs, start=1):
    if ci == 1 or ci % 250 == 0 or ci == len(configs):
        print(
            f"[{time.time()-t0:7.2f}s] config {ci}/{len(configs)}: "
            f"{signal_name}, {q_name}, w={window}, entry={entry_z}, exit={exit_z}, hold={max_hold}"
        )

    all_day_pnls = []
    all_day_hits = []
    total_trades = 0

    for d in days:
        trades = strict_backtest_one_day(
            mid=mid_by_day[int(d)],
            bid=bid_by_day[int(d)],
            ask=ask_by_day[int(d)],
            signal_func=signal_func,
            q_base=q,
            window=window,
            entry_z=entry_z,
            exit_z=exit_z,
            max_hold=max_hold,
        )

        day_summary = summarize_day_trades(trades)
        day_pnl = day_summary["day_pnl"]
        trade_count = day_summary["trade_count"]

        all_day_pnls.append(day_pnl)
        if trade_count > 0 and np.isfinite(day_summary["hit_rate"]):
            all_day_hits.append(day_summary["hit_rate"])

        total_trades += trade_count

        day_row = {
            "signal": signal_name,
            "q_name": q_name,
            "q_trade": list(q),
            "day": int(d),
            "window": window,
            "entry_z": entry_z,
            "exit_z": exit_z,
            "max_hold": max_hold,
            **day_summary,
        }
        day_rows.append(day_row)

        for t in trades:
            trade_rows.append({
                "signal": signal_name,
                "q_name": q_name,
                "day": int(d),
                "window": window,
                "entry_z": entry_z,
                "exit_z": exit_z,
                "max_hold": max_hold,
                **t,
            })

    all_day_pnls = np.array(all_day_pnls, dtype=float)

    total_pnl = float(np.sum(all_day_pnls))
    mean_day_pnl = float(np.mean(all_day_pnls))
    min_day_pnl = float(np.min(all_day_pnls))
    max_day_pnl = float(np.max(all_day_pnls))
    positive_days = int(np.sum(all_day_pnls > 0))
    active_days = int(np.sum(np.array([
        r["trade_count"] for r in day_rows[-len(days):]
    ]) > 0))

    one_day_dependency = float(max_day_pnl / total_pnl) if total_pnl > 0 else np.inf
    mean_hit_rate = float(np.mean(all_day_hits)) if all_day_hits else np.nan
    min_hit_rate = float(np.min(all_day_hits)) if all_day_hits else np.nan
    pnl_per_trade = float(total_pnl / total_trades) if total_trades else np.nan

    robust_pass = (
        positive_days == len(days)
        and total_trades >= 5
        and min_day_pnl > 0
        and one_day_dependency < 0.70
    )

    # Similar robust score idea: reward total, min day, hit rate; penalise dependency.
    robust_score = (
        total_pnl
        + 2.0 * min_day_pnl
        + 5000.0 * (0 if np.isnan(mean_hit_rate) else mean_hit_rate)
        - 10000.0 * max(0.0, one_day_dependency - 0.50)
    )

    summary_rows.append({
        "signal": signal_name,
        "q_name": q_name,
        "q_trade": list(q),
        "net_position": float(np.sum(q)),
        "gross_position": float(np.sum(np.abs(q))),
        "window": window,
        "entry_z": entry_z,
        "exit_z": exit_z,
        "max_hold": max_hold,
        "days": len(days),
        "active_days": active_days,
        "positive_days": positive_days,
        "total_pnl": total_pnl,
        "mean_day_pnl": mean_day_pnl,
        "min_day_pnl": min_day_pnl,
        "max_day_pnl": max_day_pnl,
        "total_trades": total_trades,
        "mean_hit_rate": mean_hit_rate,
        "min_hit_rate": min_hit_rate,
        "pnl_per_trade": pnl_per_trade,
        "one_day_dependency": one_day_dependency,
        "robust_pass": robust_pass,
        "robust_score": robust_score,
    })


summary_df = pd.DataFrame(summary_rows)
day_df = pd.DataFrame(day_rows)
trade_df = pd.DataFrame(trade_rows)

summary_df = summary_df.sort_values(
    ["robust_pass", "robust_score", "total_pnl"],
    ascending=[False, False, False]
).reset_index(drop=True)

robust_df = summary_df[summary_df["robust_pass"]].copy().reset_index(drop=True)

summary_path = f"{OUTDIR}/pod_confirm_summary.csv"
robust_path = f"{OUTDIR}/pod_confirm_robust_only.csv"
day_path = f"{OUTDIR}/pod_confirm_day_breakdown.csv"
trade_path = f"{OUTDIR}/pod_confirm_trades.csv"

summary_df.to_csv(summary_path, index=False)
robust_df.to_csv(robust_path, index=False)
day_df.to_csv(day_path, index=False)
trade_df.to_csv(trade_path, index=False)

print("\nSaved:")
print(summary_path)
print(robust_path)
print(day_path)
print(trade_path)

print("\nTop confirmation summary:")
display(summary_df.head(50))

print("\nRobust only:")
display(robust_df.head(50))

best = summary_df.iloc[0]
print("\nBest config:")
display(best.to_frame().T)

mask_best_day = (
    (day_df["signal"] == best["signal"])
    & (day_df["q_name"] == best["q_name"])
    & (day_df["window"] == best["window"])
    & (day_df["entry_z"] == best["entry_z"])
    & (day_df["exit_z"] == best["exit_z"])
    & (day_df["max_hold"] == best["max_hold"])
)

print("\nBest config day breakdown:")
display(day_df[mask_best_day].sort_values("day"))

mask_best_trade = (
    (trade_df["signal"] == best["signal"])
    & (trade_df["q_name"] == best["q_name"])
    & (trade_df["window"] == best["window"])
    & (trade_df["entry_z"] == best["entry_z"])
    & (trade_df["exit_z"] == best["exit_z"])
    & (trade_df["max_hold"] == best["max_hold"])
)

print("\nBest config trades:")
display(trade_df[mask_best_trade].sort_values(["day", "entry_idx"]).head(200))


# ----------------------------
# Compact q-variant comparison:
# best result per q_name, signal
# ----------------------------

q_compare = (
    summary_df
    .sort_values(["q_name", "signal", "robust_pass", "robust_score"], ascending=[True, True, False, False])
    .groupby(["q_name", "signal"], as_index=False)
    .head(1)
    .sort_values(["robust_pass", "robust_score", "total_pnl"], ascending=[False, False, False])
    .reset_index(drop=True)
)

print("\nBest per q variant and signal:")
display(q_compare.head(100))

q_compare.to_csv(f"{OUTDIR}/pod_confirm_best_per_q_signal.csv", index=False)

Days: [np.int64(2), np.int64(3), np.int64(4)]
Day 2: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Day 3: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Day 4: mid=(10000, 5), bid=(10000, 5), ask=(10000, 5)
Total configs: 13200
[   0.00s] config 1/13200: pn_vs_slc_logratio, raw_current_10, w=2500, entry=1.25, exit=0.0, hold=1000
[   3.03s] config 250/13200: pn_vs_slc_logratio, raw_current_10, w=6000, entry=1.25, exit=0.5, hold=1500
[   5.88s] config 500/13200: pn_vs_slc_logratio, raw_current_5, w=4000, entry=1.5, exit=0.0, hold=4000
[   8.70s] config 750/13200: pn_vs_slc_logratio, raw_current_5, w=7500, entry=1.5, exit=0.75, hold=1500
[  11.62s] config 1000/13200: pn_vs_slc_logratio, raw_current_15, w=5000, entry=1.75, exit=0.25, hold=4000
[  14.57s] config 1250/13200: pn_vs_slc_logratio, neutral_6_9, w=2500, entry=2.0, exit=0.0, hold=1500
[  17.45s] config 1500/13200: pn_vs_slc_logratio, neutral_6_9, w=6000, entry=2.0, exit=0.5, hold=4000
[  20.32s] config 1750/13200: pn_vs_slc_logra

,signal,q_name,q_trade,net_position,gross_position,window,entry_z,exit_z,max_hold,days,...,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
0,pn_vs_slc_spread,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,2500,3,...,31406.000000,29058.0,35022.0,25,0.878307,0.857143,3768.720000,0.371712,True,156725.534392
1,pn_vs_slc_spread,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,4000,3,...,31406.000000,29058.0,35022.0,25,0.878307,0.857143,3768.720000,0.371712,True,156725.534392
2,pn_vs_slc_ratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,2500,3,...,30016.000000,26112.0,34734.0,24,0.873677,0.857143,3752.000000,0.385728,True,146640.386243
3,pn_vs_slc_ratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,4000,3,...,30016.000000,26112.0,34734.0,24,0.873677,0.857143,3752.000000,0.385728,True,146640.386243
4,pn_vs_slc_logratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,2500,3,...,29636.000000,26184.0,34716.0,24,0.873677,0.857143,3704.500000,0.390471,True,145644.386243
5,pn_vs_slc_logratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,4000,3,...,29636.000000,26184.0,34716.0,24,0.873677,0.857143,3704.500000,0.390471,True,145644.386243
6,pn_vs_slc_spread,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,1500,3,...,28784.000000,23640.0,35022.0,26,0.847222,0.777778,3321.230769,0.405573,True,137868.111111
7,pn_vs_slc_logratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.75,0.25,2500,3,...,27112.000000,25182.0,30810.0,18,0.896825,0.833333,4518.666667,0.378799,True,136184.126984
8,pn_vs_slc_logratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.75,0.25,4000,3,...,27112.000000,25182.0,30810.0,18,0.896825,0.833333,4518.666667,0.378799,True,136184.126984
9,pn_vs_slc_ratio,raw_current_15,"[-15, -15, 15, 15, -15]",-15.0,75.0,2500,1.25,0.00,2500,3,...,28875.000000,22650.0,33870.0,24,0.830688,0.714286,3609.375000,0.390996,True,136078.439153



Robust only:


,signal,q_name,q_trade,net_position,gross_position,window,entry_z,exit_z,max_hold,days,...,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
0,pn_vs_slc_spread,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,2500,3,...,31406.000000,29058.0,35022.0,25,0.878307,0.857143,3768.720000,0.371712,True,156725.534392
1,pn_vs_slc_spread,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,4000,3,...,31406.000000,29058.0,35022.0,25,0.878307,0.857143,3768.720000,0.371712,True,156725.534392
2,pn_vs_slc_ratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,2500,3,...,30016.000000,26112.0,34734.0,24,0.873677,0.857143,3752.000000,0.385728,True,146640.386243
3,pn_vs_slc_ratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,4000,3,...,30016.000000,26112.0,34734.0,24,0.873677,0.857143,3752.000000,0.385728,True,146640.386243
4,pn_vs_slc_logratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,2500,3,...,29636.000000,26184.0,34716.0,24,0.873677,0.857143,3704.500000,0.390471,True,145644.386243
5,pn_vs_slc_logratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,4000,3,...,29636.000000,26184.0,34716.0,24,0.873677,0.857143,3704.500000,0.390471,True,145644.386243
6,pn_vs_slc_spread,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,1500,3,...,28784.000000,23640.0,35022.0,26,0.847222,0.777778,3321.230769,0.405573,True,137868.111111
7,pn_vs_slc_logratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.75,0.25,2500,3,...,27112.000000,25182.0,30810.0,18,0.896825,0.833333,4518.666667,0.378799,True,136184.126984
8,pn_vs_slc_logratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.75,0.25,4000,3,...,27112.000000,25182.0,30810.0,18,0.896825,0.833333,4518.666667,0.378799,True,136184.126984
9,pn_vs_slc_ratio,raw_current_15,"[-15, -15, 15, 15, -15]",-15.0,75.0,2500,1.25,0.00,2500,3,...,28875.000000,22650.0,33870.0,24,0.830688,0.714286,3609.375000,0.390996,True,136078.439153



Best config:


...

,signal,q_name,q_trade,day,window,entry_z,exit_z,max_hold,trade_count,day_pnl,hit_rate,avg_trade_pnl,median_trade_pnl
20406,pn_vs_slc_spread,neutral_12_18,"[-12, -12, 18, 18, -12]",2,2500,1.25,0.0,2500,9,30138.0,0.888889,3348.666667,3936.0
20407,pn_vs_slc_spread,neutral_12_18,"[-12, -12, 18, 18, -12]",3,2500,1.25,0.0,2500,9,35022.0,0.888889,3891.333333,3798.0
20408,pn_vs_slc_spread,neutral_12_18,"[-12, -12, 18, 18, -12]",4,2500,1.25,0.0,2500,7,29058.0,0.857143,4151.142857,6642.0



Best config trades:


,signal,q_name,day,window,entry_z,exit_z,max_hold,entry_idx,exit_idx,entry_ts,exit_ts,side,entry_z_seen,exit_z_seen,hold,exec_pnl,q_trade,position
59443,pn_vs_slc_spread,neutral_12_18,2,2500,1.25,0.0,2500,2838,4396,283800,439600,1,-1.252376,0.012851,1558,2106.0,"[-12, -12, 18, 18, -12]","[-12.0, -12.0, 18.0, 18.0, -12.0]"
59444,pn_vs_slc_spread,neutral_12_18,2,2500,1.25,0.0,2500,4863,6736,486300,673600,-1,1.294308,-0.016852,1873,-1818.0,"[-12, -12, 18, 18, -12]","[12.0, 12.0, -18.0, -18.0, 12.0]"
59445,pn_vs_slc_spread,neutral_12_18,2,2500,1.25,0.0,2500,6896,7551,689600,755100,1,-1.267575,0.018015,655,5232.0,"[-12, -12, 18, 18, -12]","[-12.0, -12.0, 18.0, 18.0, -12.0]"
59446,pn_vs_slc_spread,neutral_12_18,2,2500,1.25,0.0,2500,7636,7990,763600,799000,1,-1.297748,0.030079,354,3138.0,"[-12, -12, 18, 18, -12]","[-12.0, -12.0, 18.0, 18.0, -12.0]"
59447,pn_vs_slc_spread,neutral_12_18,2,2500,1.25,0.0,2500,8108,8430,810800,843000,1,-1.256278,0.010196,322,3936.0,"[-12, -12, 18, 18, -12]","[-12.0, -12.0, 18.0, 18.0, -12.0]"
59448,pn_vs_slc_spread,neutral_12_18,2,2500,1.25,0.0,2500,8494,8602,849400,860200,1,-1.298697,0.047666,108,5388.0,"[-12, -12, 18, 18, -12]","[-12.0, -12.0, 18.0, 18.0, -12.0]"
59449,pn_vs_slc_spread,neutral_12_18,2,2500,1.25,0.0,2500,8682,9174,868200,917400,1,-1.283532,0.076521,492,2064.0,"[-12, -12, 18, 18, -12]","[-12.0, -12.0, 18.0, 18.0, -12.0]"
59450,pn_vs_slc_spread,neutral_12_18,2,2500,1.25,0.0,2500,9273,9465,927300,946500,-1,1.338873,-0.016540,192,5178.0,"[-12, -12, 18, 18, -12]","[12.0, 12.0, -18.0, -18.0, 12.0]"
59451,pn_vs_slc_spread,neutral_12_18,2,2500,1.25,0.0,2500,9707,9902,970700,990200,-1,1.267891,-0.053155,195,4914.0,"[-12, -12, 18, 18, -12]","[12.0, 12.0, -18.0, -18.0, 12.0]"
59452,pn_vs_slc_spread,neutral_12_18,3,2500,1.25,0.0,2500,2499,2859,249900,285900,-1,1.573661,-0.036734,360,11760.0,"[-12, -12, 18, 18, -12]","[12.0, 12.0, -18.0, -18.0, 12.0]"



Best per q variant and signal:


,signal,q_name,q_trade,net_position,gross_position,window,entry_z,exit_z,max_hold,days,...,mean_day_pnl,min_day_pnl,max_day_pnl,total_trades,mean_hit_rate,min_hit_rate,pnl_per_trade,one_day_dependency,robust_pass,robust_score
0,pn_vs_slc_spread,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,2500,3,...,31406.000000,29058.0,35022.0,25,0.878307,0.857143,3768.720000,0.371712,True,1.567255e+05
1,pn_vs_slc_ratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,2500,3,...,30016.000000,26112.0,34734.0,24,0.873677,0.857143,3752.000000,0.385728,True,1.466404e+05
2,pn_vs_slc_logratio,neutral_12_18,"[-12, -12, 18, 18, -12]",0.0,72.0,2500,1.25,0.00,2500,3,...,29636.000000,26184.0,34716.0,24,0.873677,0.857143,3704.500000,0.390471,True,1.456444e+05
3,pn_vs_slc_ratio,raw_current_15,"[-15, -15, 15, 15, -15]",-15.0,75.0,2500,1.25,0.00,2500,3,...,28875.000000,22650.0,33870.0,24,0.830688,0.714286,3609.375000,0.390996,True,1.360784e+05
4,pn_vs_slc_spread,raw_current_15,"[-15, -15, 15, 15, -15]",-15.0,75.0,2500,1.25,0.00,2500,3,...,29110.000000,21900.0,33150.0,25,0.793651,0.666667,3493.200000,0.379595,True,1.350983e+05
5,pn_vs_slc_logratio,raw_current_15,"[-15, -15, 15, 15, -15]",-15.0,75.0,2500,1.25,0.00,2500,3,...,28500.000000,21240.0,33900.0,24,0.830688,0.714286,3562.500000,0.396491,True,1.321334e+05
6,pn_vs_slc_spread,neutral_10_15,"[-10, -10, 15, 15, -10]",0.0,60.0,2500,1.25,0.00,2500,3,...,26171.666667,24215.0,29185.0,25,0.878307,0.857143,3140.600000,0.371712,True,1.313365e+05
7,pn_vs_slc_ratio,neutral_10_15,"[-10, -10, 15, 15, -10]",0.0,60.0,2500,1.25,0.00,2500,3,...,25013.333333,21760.0,28945.0,24,0.873677,0.857143,3126.666667,0.385728,True,1.229284e+05
8,pn_vs_slc_logratio,neutral_10_15,"[-10, -10, 15, 15, -10]",0.0,60.0,2500,1.25,0.00,2500,3,...,24696.666667,21820.0,28930.0,24,0.873677,0.857143,3087.083333,0.390471,True,1.220984e+05
9,pn_vs_slc_spread,no_cotton_10_15,"[-10, -10, 15, 15, 0]",10.0,50.0,4000,1.50,0.00,2500,3,...,23255.000000,20105.0,27030.0,12,0.944444,0.833333,5813.750000,0.387444,True,1.146972e+05
